<a href="https://colab.research.google.com/github/tuonglab/BIOL3003_workshop/blob/master/notebook/BIOL3003_cell_cell_comunication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UQ BIOL3003 scRNA-seq cell-cell communication analysis workshop!

Before we dive into the demo, let's first install the necessary packages.

In [ ]:
# setup the notebook
!pip install -qqq scanpy[leiden] bbknn celltypist cellphonedb ktplotspy
# then clone the repository so that we have all the data and notebooks ready to go
!git clone https://github.com/tuonglab/BIOL3003_workshop.git

# Single-cell RNA seq analysis Demo

This demo will show you the common steps involved to get you started on single cell analysis in Python using [`Scanpy`](https://scanpy.readthedocs.io/en/stable/), the toolkit for analysing single-cell gene expression data.

<a href="https://scanpy.readthedocs.io/en/stable/"><img src="https://scanpy.readthedocs.io/en/stable/_static/Scanpy_Logo_BrightFG.svg" alt="anndata_schema" width="100">


## Preprocessing and Quality Control

First, import packages needed for single-cell RNA seq analysis.

In [ ]:
import os

import scanpy as sc
import pandas as pd

# change to working directory
os.chdir("BIOL3003_workshop")

### Reading in files for analysis

For this demo, we have already saved the starting raw datafile as an `.h5ad` file which is a common file format used in single-cell analysis. You can read in the file using the `read_h5ad` function from [`anndata`](https://anndata.readthedocs.io/) package.

This file contains the raw counts of the cells and genes, as well as the metadata associated with the cells and genes.

The file is saved in the `data` folder.


<a href="https://anndata.readthedocs.io/"><img src="https://raw.githubusercontent.com/scverse/anndata/main/docs/_static/img/anndata_schema.svg" alt="anndata_schema" width="500">


The dataset we will be demo-ing today is on the human prostate.

<a href="https://www.frontiersin.org/journals/endocrinology/articles/10.3389/fendo.2022.1006101/full"><img src="https://www.frontiersin.org/files/Articles/1006101/fendo-13-1006101-HTML-r1/image_m/fendo-13-1006101-g001.jpg" alt="human prostate schema" width="500">


In [ ]:
import scanpy as sc

adata = sc.read_h5ad("data/prostate_demo.h5ad")
# adata = sc.read_h5ad("../data/prostate_demo.h5ad")
adata

## Standard Quality control

A very common QC step is to assess the mitochondrial content.

High mitochondrial content is often associated with poor quality cells. We can calculate the percentage of mitochondrial genes in each cell and plot it.

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=[
        "mt",
    ],
    inplace=True,
    log1p=True,
)

In [ ]:
sc.pl.violin(
    adata,
    [
        "n_genes_by_counts",  # the number of genes expressed in the count matrix
        "total_counts",  # the total umi counts per cell
        "pct_counts_mt",  # the percentage of counts in mitochondrial/ribosomal genes
    ],
    jitter=0.4,
    multi_panel=True,
)

Continue processing with "good" cells only..

In [ ]:
# filter cells if they do not express at least 200 genes
sc.pp.filter_cells(adata, min_genes=200)
# filter genes if they are expressed in at least 3 cells
sc.pp.filter_genes(adata, min_cells=3)
# always check after you have done some filtering to ensure that you are happy with the results
adata

## Normalisation

In [ ]:
# Normalise (library-size correct) the data matrix 𝐗 to 10,000 counts per cell, so that information become comparable between cells.
sc.pp.normalize_total(adata, target_sum=1e4)

# Logarithmise the data:
sc.pp.log1p(adata)

## Highly Variable Feature/Gene selection

Identify and inspect highly-variable genes

In [ ]:
# (Expects logarithimised data)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
# stash the normalised counts in .raw, before we subset to only highly variable genes
adata.raw = adata

## Dimensionality Reduction

### Step 1: Subset to only highly variable genes

In [ ]:
# Actually do the filtering for PCA
adata = adata[:, adata.var["highly_variable"] == True].copy()
adata

### Step 2: Regress out effects of "total_counts" per cell and percentage of mitochondrial genes expressed

In [ ]:
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])

### Step 3: Scale each gene to unit variance. Clip values exceeding standard deviation of 10.

In [ ]:
sc.pp.scale(adata, max_value=10)

### Step 4: Perform Principal Component Analysis (PCA)

In [ ]:
sc.tl.pca(adata)
# visualise the variance contribution by each PC
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

### Step 5: Compute neighbourhood graph

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

### Step 6: Embed the neighbourhood graph using UMAP

UMAP stands for Uniform Manifold Approximation and Projection. It is a non-linear dimensionality reduction technique that is well-suited for preserving local structure in high-dimensional data.

In [ ]:
sc.tl.umap(adata)

#### Visualise UMAP:

In [ ]:
sc.pl.umap(adata, color=["patient", "group", "age"])

### Reapeat with batch correction!

`bbknn` is a lightweight batch correction tool that corrects for batch effects in single-cell RNA-seq data. It is based on the k-nearest neighbors (KNN) algorithm and is designed to be fast and scalable.

In [ ]:
# repeat the steps until umap
sc.external.pp.bbknn(adata, batch_key="patient")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["patient"])

### Step 6: Clustering

We will use the `leiden` algorithm to cluster the cells into different groups. It is a graph-based clustering algorithm that is very popular in single-cell analysis. It is based on optimizing a modularity function that is used to detect communities in networks. It has a resolution parameter that can be tuned to get different levels of granularity in the clustering.

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color="leiden")

### Step 7: Cell type annotation

One of the most important steps for scRNA-seq analysis is to perform cell-type annotation. If we don't do this, we can't interpret the data.

Today, we will use a tool called `CellTypist` to help us automate this process.

[`CellTypist`](https://www.celltypist.org/) is a tool that uses a machine learning model to predict cell types based on marker genes. It is a very powerful tool that can be used to predict cell types in single-cell data.

First, install `celltypist` and prepare the data.

In [ ]:
# make a copy of the log1p normalised data for celltypist
for_celltypist = adata.raw.to_adata()

In [ ]:
# load up celltypist
import celltypist
from celltypist import models

# Download a specific model, for example, `Immune_All_Low.pkl`.
models.download_models(model="Immune_All_Low.pkl")

Run celltypist on our data and allow it to predict labels on each single cell with all the specifications needed.

In [ ]:
predictions = celltypist.annotate(
    for_celltypist, model="Immune_All_Low.pkl", majority_voting=True
)
# transfer the predictions back to the original adata
adata.obs["celltypist_majority_voting"] = predictions.predicted_labels.majority_voting

Visualise the data via umap with the new celltypist labels

In [ ]:
sc.pl.umap(adata, color=["celltypist_majority_voting"])

### Let's examine some genes

In [ ]:
# these are T-cell genes
marker_genes = [
    "CD4",
    "CD8B",
    "FOXP3",
    "SELL",
    "CCR7",
    "MKI67",
    "NKG7",
    "GATA3",
    "RORC",
    "CXCR5",
    "CD69",
    "GZMK",
]
sc.pl.umap(adata, color=marker_genes, ncols=3)

In [ ]:
sc.pl.dotplot(
    adata,
    marker_genes,
    groupby="celltypist_majority_voting",
    standard_scale="var",
    color_map="Blues",
)

## Cell-cell Receptor-Ligand Interaction Analysis

Now let's see which cells could be talking to each other!

In [ ]:
# first set up the database
from cellphonedb.utils import db_utils

db_utils.download_database(".", "v5.0.0")
cpdb_file_path = "cellphonedb.zip"
meta_file_path = "meta.txt"
counts_file_path = "for_cellphonedb.h5ad"

# cell label info
meta = (
    adata.obs["celltypist_majority_voting"]
    .reset_index(drop=False)
    .to_csv(meta_file_path, sep="\t", index=False)
)
# expression data
for_celltypist.write_h5ad(counts_file_path, compression="gzip")

In [ ]:
# run the actual analysis - can take a while
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path=cpdb_file_path,
    meta_file_path=meta_file_path,
    counts_file_path=counts_file_path,
    counts_data="hgnc_symbol",
    output_path="cpdb_results",
    iterations=100,
)

This analysis only 'predicts' since it's all still based on mRNA expression.

<img src="https://private-user-images.githubusercontent.com/26215587/466229488-f26d6ed5-4090-4e1e-9bc4-db375e49afbf.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NzgxMDk4ODcsIm5iZiI6MTc3ODEwOTU4NywicGF0aCI6Ii8yNjIxNTU4Ny80NjYyMjk0ODgtZjI2ZDZlZDUtNDA5MC00ZTFlLTliYzQtZGIzNzVlNDlhZmJmLnBuZz9YLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFLSUFWQ09EWUxTQTUzUFFLNFpBJTJGMjAyNjA1MDYlMkZ1cy1lYXN0LTElMkZzMyUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNTA2VDIzMTk0N1omWC1BbXotRXhwaXJlcz0zMDAmWC1BbXotU2lnbmF0dXJlPTAwNmQ4NGM3YmJjMTk4ZjJiOWNmMWFmMGQ4MDFjYmE2ZjU0YzRkOTkyNzk0MWNiYmNjYjVjMTk4NjdkYTg2M2MmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JnJlc3BvbnNlLWNvbnRlbnQtdHlwZT1pbWFnZSUyRnBuZyJ9.4crloOUDldWtDYaNel47uE9Bw7ze-3VuoHB5cexF-ds" alt="cpdb" width="500">

In [ ]:
# visualise - as heatmap
import ktplotspy as kpy

kpy.plot_cpdb_heatmap(
    pvals=cpdb_results["pvalues"],
    degs_analysis=False,
    figsize=(10, 10),
    title="Sum of significant interactions",
)

In [ ]:
# visualise individual interactions - can take a while
kpy.plot_cpdb(
    adata=adata,
    cell_type1="NK cell",
    cell_type2=".",
    means=cpdb_results["means"],
    pvals=cpdb_results["pvalues"],
    celltype_key="celltypist_majority_voting",
    # genes = ["IFNG", "IL6"],
    figsize=(10, 12),
    max_size=3,
    highlight_size=0.75,
    degs_analysis=False,
)

In [ ]:
kpy.plot_cpdb_chord(
    adata=adata,
    cell_type1="NK cell",
    cell_type2=".",
    means=cpdb_results["means"],
    pvals=cpdb_results["pvalues"],
    deconvoluted=cpdb_results["deconvoluted"],
    celltype_key="celltypist_majority_voting",
    interaction=["IL1A", "IFNG", "IL6"],
    link_kwargs={"direction": 1, "allow_twist": True, "r1": 95, "r2": 90},
    sector_text_kwargs={
        "color": "black",
        "size": 12,
        "r": 105,
        "adjust_rotation": True,
    },
    legend_kwargs={"loc": "center", "bbox_to_anchor": (1, 1), "fontsize": 8},
    link_offset=1,
)

# Other useful resources
For more details on this dataset that we demoed today, please checkout the original publication and data portal:

https://www.prostatecellatlas.org/

<a href="https://doi.org/10.1016/j.celrep.2021.110132"><img src="https://www.prostatecellatlas.org/assets/cover.jpg" alt="prostate_cellrep" width="200">

If you have any questions, email Kelvin at z.tuong@uq.edu.au